In [1]:
import torch
from pathlib import Path

In [2]:
# Get checkpoints files from the range 
ckpt_dir = Path("/data/ratna/aug/randstain_strong/eval") # RSG - changed from /data/OM_checkpoints , /data/ratna_retrain/eval
print(ckpt_dir)
steps = list(range(87500, 137501, 12500))  # RSG - change this for a new range
ckpt_paths = [ ckpt_dir / f"training_{step}" / "teacher_checkpoint.pth" for step in steps]
print(f"Checkpoints to average: {steps}")
print(f"Number of paths: {len(ckpt_paths)}")

/data/ratna/aug/randstain_strong/eval
Checkpoints to average: [87500, 100000, 112500, 125000, 137500]
Number of paths: 5


In [3]:
# Filter and load checkpoints
state_dicts = [torch.load(str(p), map_location='cpu') for p in ckpt_paths if p.exists()]
print(f"Loaded {len(state_dicts)} checkpoints")

Loaded 5 checkpoints


In [4]:
teacher_dicts = [sd["teacher"] for sd in state_dicts]

In [5]:
print(len(teacher_dicts), teacher_dicts[0])

5 OrderedDict([('backbone.cls_token', tensor([[[-0.0268,  0.0013,  0.0039,  ...,  0.0022, -0.0225,  0.0032]]])), ('backbone.pos_embed', tensor([[[-0.0268,  0.0013,  0.0040,  ...,  0.0022, -0.0225,  0.0032],
         [ 0.1081,  0.0125, -0.0567,  ..., -0.0027, -0.0837,  0.0178],
         [-0.0065, -0.0030,  0.0173,  ...,  0.0084,  0.0229,  0.0059],
         ...,
         [-0.0192, -0.0064,  0.0035,  ...,  0.0054, -0.0031,  0.0003],
         [ 0.0009,  0.0063, -0.0002,  ...,  0.0089,  0.0025, -0.0066],
         [-0.0016,  0.0319, -0.0135,  ...,  0.0006, -0.0276,  0.0065]]])), ('backbone.register_tokens', tensor([[[-3.6599e-04,  2.3635e-03,  1.0969e-04,  ...,  1.8397e-04,
          -4.5217e-02,  1.7199e-03],
         [-1.9521e-01,  6.8846e-04, -1.8943e-02,  ..., -4.6818e-04,
           4.5741e-02,  4.7343e-03],
         [-1.5816e-02, -1.4095e-03, -1.0412e-01,  ..., -7.2073e-03,
           5.2381e-02,  4.5100e-03],
         [ 2.0480e-02,  4.0690e-03,  1.9130e-03,  ...,  1.5569e-03,
        

In [6]:
averaged_state_dict = {}
for key in teacher_dicts[0].keys():
    averaged_state_dict[key] = sum(td[key] for td in teacher_dicts) / len(teacher_dicts)

In [7]:
# Wrap back in the original structure
output_dict = {'teacher': averaged_state_dict}
print(f"Average dtype: {output_dict['teacher']['backbone.cls_token'].dtype}")

Average dtype: torch.float32


In [ ]:
# Save averaged checkpoint
output_dir = Path("/data/ratna/aug/randstain_strong/eval/averaged_87500_to_137500") # RSG - change this for a new range
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "teacher_checkpoint.pth"
torch.save(output_dict, output_path)
print(f"Saved to: {output_path}")

Saved to: /data/ratna/aug/randstain_medium/eval/averaged_87500_to_137500/teacher_checkpoint.pth


The End

In [ ]:
ckpt_path = Path("/data/ratna/aug/randstain_strong/eval/averaged_87500_to_137500/teacher_checkpoint.pth")
state_dict = torch.load(ckpt_path, map_location="cpu")
print(state_dict)

{'teacher': {'backbone.cls_token': tensor([[[-0.0213,  0.0003,  0.0052,  ...,  0.0014, -0.0230,  0.0011]]]), 'backbone.pos_embed': tensor([[[-0.0212,  0.0004,  0.0052,  ...,  0.0014, -0.0230,  0.0011],
         [ 0.0793, -0.0160, -0.0954,  ..., -0.0123, -0.0530,  0.0296],
         [-0.0015, -0.0020,  0.0311,  ...,  0.0118,  0.0198, -0.0001],
         ...,
         [ 0.0114,  0.0083, -0.0013,  ...,  0.0119, -0.0139,  0.0007],
         [-0.0144,  0.0002,  0.0015,  ...,  0.0214, -0.0153, -0.0047],
         [-0.0146,  0.0234, -0.0081,  ..., -0.0027,  0.0316,  0.0049]]]), 'backbone.register_tokens': tensor([[[-9.9539e-04,  9.3469e-05,  2.9562e-05,  ..., -1.9889e-04,
          -4.3152e-02, -1.5939e-06],
         [-1.8810e-01,  3.1636e-03, -2.1663e-02,  ...,  1.5610e-03,
           3.8355e-02,  3.7161e-03],
         [-2.1795e-02, -2.4008e-03, -9.9075e-02,  ..., -5.6782e-03,
           3.9684e-02,  5.1020e-03],
         [ 1.2917e-02,  3.0719e-03,  2.8867e-03,  ...,  6.8942e-06,
          -9.60